# មេរៀន ០៤ - គំរូការរចនាការប្រើប្រាស់ឧបករណ៍

ក្នុងមេរៀននេះ អ្នកនឹងរៀនពីគំរូការរចនាផ្នែក **ការប្រើប្រាស់ឧបករណ៍** សម្រាប់ភ្នាក់ងារពហុមុខងារក្នុងកម្មវិធី Microsoft Agent Framework (Python)។ ពួកយើងគ្របដណ្តប់៖

- ការបញ្ជាក់ឧបករណ៍មុខងារជាមួយ`@tool` decorator និងប៉ារ៉ាម៉ែត្រប្រភេទ typed
- ការផ្តល់ schemas ឧបករណ៍ ដើម្បីឲ្យម៉ូឌែលដឹងថាឧបករណ៍នីមួយៗធ្វើអ្វីបានខ្លះ
- ឧបករណ៍គ្រប់គ្រងការប្រតិបត្តិការឧបករណ៍ជាមួយ `approval_mode`
- ការផ្ដល់ចេញ **លទ្ធផលមានរចនាសម្ព័ន្ធ** តាមរយៈម៉ូឌែល Pydantic និង `response_format`

សេណារីយោគឺជាភ្នាក់ងារកក់ចំណតដំណើរកម្សាន្តដែលអាចស្វែងរកគោលដៅ ពិនិត្យមើលភាពមានស្រាប់ និងយកព័ត៌មានអាកាសចរណ៍បាន។


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ការបង្កើតឧបករណ៍ជាមួយ @tool Decorator

@tool decorator បម្លែងមុខងារ Python ធម្មតាទៅជាឧបករណ៍មួយដែល đại lý អាចហៅបាន។
ចំណុចសំខាន់ៗ៖

- **docstring** ក្លាយទៅជាការពិពណ៌នាឧបករណ៍ដែលម៉ូដែលឃើញ។
- **Type annotations** (រួមទាំង `Annotated` ជាមួយការពណ៌នា) កំណត់ schema របស់ឧបករណ៍។
- `approval_mode` គ្រប់គ្រងថាតើអ្នកប្រើត្រូវតែអនុម័តការហៅនីមួយៗមុនពេលវាដំណើរការ។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## បង្កើតនិយោជកជាមួយឧបករណ៍ច្រើន

ផ្តល់ឧបករណ៍ទាំងបីទៅអតិថិជន ដើម្បីឲ្យម៉ូដែលអាចអំពាវនាវឧបករណ៍ណាមួយដែលវាចាំបាច់ក្នុងការឆ្លើយសំណួររបស់អ្នកប្រើ។


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ល្បែងលំដាប់ជាមួយឧបករណ៍

ដោយកំណត់ `response_format` ទៅម៉ូដែល Pydantic អ្នកចាត់ចែងត្រូវបានបង្ខំឱ្យត្រឡប់វត្ថុ JSON ដែលមានប្រភេទល្អជាជម្រើសផ្ទាល់ជំនួសអត្ថបទដែលមានទ្រង់ទ្រាយសេរី។ វាមានប្រយោជន៍នៅពេលកូដខាងក្រោមត្រូវការប្រើលទ្ធផលដោយកម្មវិធី។


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## គំរូការអនុញ្ញាតឧបករណ៍

ប៉ារ៉ាម៉ែត្រ `approval_mode` នៅលើ `@tool` គ្រប់គ្រងថាតើការហៅឧបករណ៍ត្រូវការអនុញ្ញាតពីមនុស្សមុនពេលអនុវត្ត៖

| របៀប | អាកប្បកិរិយា |
|---|---|
| `"never_require"` | ឧបករណ៍រត់ស្វ័យប្រវត្តិ — មិនចាំបាច់បញ្ជាក់ពីអ្នកប្រើ។ |
| `"always_require"` | រាល់ការហៅត្រូវតែបានអនុម័តដោយអ្នកប្រើមុនពេលអនុវត្ត។ |

ប្រើ `"always_require"` សម្រាប់ឧបករណ៍ដែលមានផលប៉ះពាល់ក្រៅ (ឧ. ការកក់ទូតយន្តហោះ ការចំណាយកាតឥណទាន) ដើម្បីឲ្យមនុស្សនៅក្នុងដំណើរការ។


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប៖

1. **កំណត់ឧបករណ៍** ដោយប្រើ `@tool` decorator ជាមួយប៉ារ៉ាម៉ែត្រ typed និង docstrings ដែលបម្រើជាស្កេម៉ាទីឧបករណ៍។
2. **បង្កើតឧបករណ៍ច្រើន** ដើម្បីឲ្យភ្នាក់ងារអាចហៅវាក្នុងលំដាប់ ដើម្បីឆ្លើយសំណួរដែលស្មុគស្មាញ។
3. **ត្រឡប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ** ដោយផ្ទេរម៉ូឌែល Pydantic ជា `response_format`។
4. **ដាក់ការត្រួតពិនិត្យការអនុម័តឧបករណ៍** ជាមួយ `approval_mode` ដើម្បីរក្សាមនុស្សនៅក្នុងលំហូរសម្រាប់ប្រតិបត្តិការដែលមានប្រសិទ្ធភាព។

រចនាប្រភេទទាំងនេះជាគ្រឹះសម្រាប់ការបង្កើតភ្នាក់ងារដែលទុកចិត្តបាន និងគ្រោងនឹងប្រើប្រាស់ជាសាធារណៈ អាចអន្តរាកម្មជាមួយប្រព័ន្ធក្រៅបានយ៉ាងបានសុវត្ថិភាព។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
